# Momento de Retroalimentación: Redefinición de los datos


En este proyecto nos interesa entrenar modelos de aprendizaje de máquina para el dataset asignado. 

Para ello, y después de haber realizado tu EDA y en caso de ser necesario tu ETL, ahora deberás crear un nuevo dataset apropiado para ser empleado con una técnica clásica de ML.


Define una estructura para los datos, tal que extraiga información del dataset original (ya sea el raw o el preprocesado).

Explica esa estructura y sube en tu repositorio de GitHub tu nueva versión del dataset.

Programa uno de los algoritmos vistos en el módulo (o que tu profesor de módulo autorice) sin usar ninguna biblioteca o framework de aprendizaje máquina, ni de estadística avanzada. 

Lo que se busca es que implementes manualmente el algoritmo, no que importes un algoritmo ya implementado. 

Prueba tu implementación con tu set de datos y realiza algunas predicciones. Las predicciones las puedes correr en consola o las puedes implementar con una interfaz gráfica apoyándote en los visto en otros módulos.

Tu implementación debe de poder correr por separado solamente con un compilador, no debe de depender de un IDE o de un “notebook”. Por ejemplo, si programas en Python, tu implementación final se espera que esté en un archivo .py no en un Jupyter Notebook.

Después de la entrega intermedia se te darán correcciones que puedes incluir en tu entrega final.

In [2]:
from dotenv import load_dotenv
import numpy as np
import os





## Dataset Original

Del análisis exploratorio obtuvimos una estructura dividida en dos archivos que incluían los features en una matriz X y un vector y con las clases de los ejercicios extraidos directamente de los datos Raw que nos permitió evaluar todos los ejercicios incluyendo el archivo corrupto del ejercicio 14.

In [11]:
load_dotenv()

X_PATH = os.getenv('X_PATH')
Y_PATH = os.getenv('Y_PATH')
X = np.load(X_PATH)
y = np.load(Y_PATH)

Para reducir tiempos de computo dentro de esta actividad exploratoria de modelos sol se seleccionaron los primeros 7 tipos de ejercicios.

In [12]:
ejercicios_seleccionados = [0, 1, 2, 3, 4, 5, 6]  # o los nombres/códigos que correspondan

mask = np.isin(y, ejercicios_seleccionados)

X = X[mask]
y = y[mask]

print("Nuevo shape de X:", X.shape)
print("Distribución de clases:", np.unique(y, return_counts=True))

Nuevo shape de X: (35190, 200, 12)
Distribución de clases: (array([0, 1, 2, 3, 4, 5, 6]), array([4176, 3816, 5598, 4230, 5274, 5634, 6462]))


esta nueva matriz X reducida será con la que operemos a partir de ahora, el siguiente paso incluye una separación de los datos en training y test sets. Posterior a este se realiza un escalamiento de los datos.

In [13]:
def train_test_split_manual(X, y, test_ratio=0.2, seed=42):
    np.random.seed(seed)
    n = X.shape[0]
    indices = np.arange(n)
    np.random.shuffle(indices)

    n_test = int(n * test_ratio)
    test_idx = indices[:n_test]
    train_idx = indices[n_test:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split_manual(X, y)

In [ ]:
def fit_scaler(X_train):
    mean = X_train.mean(axis=(0, 1))  
    std = X_train.std(axis=(0, 1)) + 1e-8
    return mean, std

def apply_scaler(X, mean, std):
    return (X - mean) / std

mean, std = fit_scaler(X_train)
X_train_scaled = apply_scaler(X_train, mean, std)
X_test_scaled = apply_scaler(X_test, mean, std)


posterior al escalamiento procedemos a extraer las features de la totalidad de la señal; Para este primer acercamiento se decidió trabajar con 5 metricas para cada uno de los 12 canales dando así 60 nuevas features que serán las variables de entrada para el modelo, estas métricas incluyen la media, desviación estándar, valores mínimo y máximo y la cantidad de cruces de la señal.

In [17]:
def extract_features(X):
    n_samples, n_timesteps, n_channels = X.shape
    features = []

    for i in range(n_samples):
        sample_feats = []
        for c in range(n_channels):
            señal = X[i, :, c]
            media = np.mean(señal)
            desv = np.std(señal)
            minimo = np.min(señal)
            maximo = np.max(señal)
            cruces = np.sum(np.diff(np.sign(señal)) != 0)

            sample_feats.extend([media, desv, minimo, maximo, cruces])
        features.append(sample_feats)

    return np.array(features)


X_train_feats = extract_features(X_train_scaled)
X_test_feats = extract_features(X_test_scaled)

np.save('../data/X_train_feats.npy', X_train_feats)
np.save('../data/X_test_feats.npy', X_test_feats)
np.save('../data/y_train.npy', y_train)
np.save('../data/y_test.npy', y_test)

esto nos deja con la nueva dimensión de nuestros conjuntos con las nuevas 60 variables

In [21]:
print(f"las dimensiones del nuevo training set son: {X_train_feats.shape}")
print(f"las dimensiones del nuevo test set son: {X_train_feats.shape}")


las dimensiones del nuevo training set son: (28152, 60)
las dimensiones del nuevo test set son: (28152, 60)


Finalmente la evaluación con un modelo de KNN implementado a paso

In [19]:
def distancia_euclidiana(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def knn_predict(X_train, y_train, x_query, k=5):
    distancias = [distancia_euclidiana(x_query, x_train_i) for x_train_i in X_train]
    k_indices = np.argsort(distancias)[:k]
    k_labels = y_train[k_indices]
    valores, conteos = np.unique(k_labels, return_counts=True)
    return valores[np.argmax(conteos)]

def matriz_confusion(y_true, y_pred, clases):
    n_clases = len(clases)
    clase_a_idx = {c: i for i, c in enumerate(clases)}
    matriz = np.zeros((n_clases, n_clases), dtype=int)

    for real, pred in zip(y_true, y_pred):
        i = clase_a_idx[real]
        j = clase_a_idx[pred]
        matriz[i, j] += 1

    return matriz  # filas = clase real, columnas = clase predicha

def metricas_por_clase(matriz, clases):
    n_clases = len(clases)
    precision = np.zeros(n_clases)
    recall = np.zeros(n_clases)
    f1 = np.zeros(n_clases)

    for i in range(n_clases):
        vp = matriz[i, i]
        fp = np.sum(matriz[:, i]) - vp
        fn = np.sum(matriz[i, :]) - vp

        precision[i] = vp / (vp + fp) if (vp + fp) > 0 else 0.0
        recall[i] = vp / (vp + fn) if (vp + fn) > 0 else 0.0
        f1[i] = (2 * precision[i] * recall[i] / (precision[i] + recall[i])
                 if (precision[i] + recall[i]) > 0 else 0.0)

    return precision, recall, f1

def knn_evaluate(X_train, y_train, X_test, y_test, k=5):
    predicciones = [knn_predict(X_train, y_train, x, k) for x in X_test]
    predicciones = np.array(predicciones, dtype=y_train.dtype)

    # --- Métricas ---
    accuracy = np.mean(predicciones == y_test)
    clases = np.unique(np.concatenate([y_train, y_test]))
    matriz = matriz_confusion(y_test, predicciones, clases)
    precision, recall, f1 = metricas_por_clase(matriz, clases)

    resultados = {
        "accuracy": accuracy,
        "matriz_confusion": matriz,
        "clases": clases,
        "precision_por_clase": precision,
        "recall_por_clase": recall,
        "f1_por_clase": f1,
        "precision_macro": np.mean(precision),
        "recall_macro": np.mean(recall),
        "f1_macro": np.mean(f1),
        "predicciones": predicciones,
    }

    return resultados


# --- Uso ---
resultados = knn_evaluate(X_train_feats, y_train, X_test_feats, y_test, k=3)

print("Accuracy:", resultados["accuracy"])
print("F1 macro:", resultados["f1_macro"])
print("\nMatriz de confusión (filas=real, columnas=predicho):")
print(resultados["matriz_confusion"])

print("\nPor clase:")
for i, c in enumerate(resultados["clases"]):
    print(f"Clase {c}: precision={resultados['precision_por_clase'][i]:.3f}, "
          f"recall={resultados['recall_por_clase'][i]:.3f}, "
          f"f1={resultados['f1_por_clase'][i]:.3f}")

Accuracy: 0.9073600454674623
F1 macro: 0.9063607007846068

Matriz de confusión (filas=real, columnas=predicho):
[[ 831    1    3    6    1    1    0]
 [   4  701   10   57    2    2    3]
 [   6   46 1035   27    2    3    2]
 [   1    6    5  812    3    0    1]
 [   2    9   16  218  795    5    6]
 [   0    4    9   39   12  983   40]
 [   2    9   19   28   19   23 1229]]

Por clase:
Clase 0: precision=0.982, recall=0.986, f1=0.984
Clase 1: precision=0.903, recall=0.900, f1=0.902
Clase 2: precision=0.943, recall=0.923, f1=0.933
Clase 3: precision=0.684, recall=0.981, f1=0.806
Clase 4: precision=0.953, recall=0.756, f1=0.844
Clase 5: precision=0.967, recall=0.904, f1=0.934
Clase 6: precision=0.959, recall=0.925, f1=0.942


Obtenemos la matriz de confusion que contiene los resultados de las predicciones y las metricas individuales por clase donde podemos ver que las clases en donde la precisión de la clase 3 y recall de la clase 4 son mas bajas y el f1 score de ambas tambien es bajo, viendo la matriz podemos concluir que al menos con las features seleccionadas la clase 3 y 4 son un poco más dificiles de distinguir del resto 

En general el modelo obtiene un accuracy del 90% lo cuál nos indica que obtuvimos una buena capacidad de predicción en general, aunque nuevamente esto tambien deberá probarse incluyendo el resto de ejercicios para confirmar que KNN es un modelo apropiado para la predicción de los ejercicios.